# Classificação Binária com Dados Desbalanceados

O problema original de regressão de precipitação foi convertido em uma **classificação binária**, atribuindo classe `0` às observações sem chuva e classe `1` às observações com precipitação.

Foi utilizado um **Gradient Boosting Classifier** como modelo base e comparados quatro cenários:

* dados originais, sem balanceamento;
* **Random Undersampling**;
* **Random Oversampling**;
* alteração do **limiar de decisão para 0,3**.

O desempenho das abordagens é comparado por meio de **matrizes de confusão** e métricas como **precision, recall e F1-score**, buscando avaliar como diferentes estratégias afetam principalmente a identificação da classe minoritária.


#### Imports

In [9]:
import numpy as np
import pickle
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

#### Carregando e Convertendo dados

In [2]:
filename = "../data/A652.pickle"
with open(filename, 'rb') as f:
    X_train, y_train, X_val, y_val, X_test, y_test = pickle.load(f)    

# === 2. Convertendo para classificação binária ===
# Se y == 0 -> 0; senão -> 1
y_train_bin = (y_train > 0).astype(int)
y_val_bin   = (y_val > 0).astype(int)
y_test_bin  = (y_test > 0).astype(int)


In [3]:
print("y_train_1:", y_train_bin.sum())  # Conta quantos exemplos são da classe 1
print("y_train_0", y_train_bin.shape) # Conta quantos exemplos são da classe 0
print("y_val_1:", y_val_bin.sum())  
print("y_val shape:", y_val_bin.shape)
print("y_test_1:", y_test_bin.sum())  
print("y_test shape:", y_test_bin.shape)


y_train_1: 754
y_train_0 (10012, 1)
y_val_1: 229
y_val shape: (2506, 1)
y_test_1: 813
y_test shape: (9582, 1)


#### Função auxiliar para treinar e avaliar

In [4]:

def avaliar_modelo(modelo, X_train, y_train, X_test, y_test, titulo): # Função auxiliar para treinar e avaliar
    modelo.fit(X_train, y_train) # treina o modelo
    y_pred = modelo.predict(X_test) # faz previsões no conjunto de teste
    print(f"\n=== {titulo} ===") # imprime o título da avaliação
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred, zero_division=0))

#### Modelo sem balanceamento

In [5]:
modelo_base = GradientBoostingClassifier()
avaliar_modelo(modelo_base, X_train, y_train_bin, X_test, y_test_bin, "Modelo sem balanceamento")

c:\Users\joaopjpc\Desktop\Machine-Learning-\.MLvenv\lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



=== Modelo sem balanceamento ===
Matriz de Confusão:
[[8675   94]
 [ 514  299]]
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.94      0.99      0.97      8769
           1       0.76      0.37      0.50       813

    accuracy                           0.94      9582
   macro avg       0.85      0.68      0.73      9582
weighted avg       0.93      0.94      0.93      9582



#### Undersampling 

In [6]:
undersampler = RandomUnderSampler(random_state=42) # Undersampling para balancear as classes 
X_under, y_under = undersampler.fit_resample(X_train, y_train_bin) # aplica o undersampling que reduz a classe majoritária
modelo_under = GradientBoostingClassifier()
avaliar_modelo(modelo_under, X_under, y_under, X_test, y_test_bin, "Modelo com Undersampling")


=== Modelo com Undersampling ===
Matriz de Confusão:
[[7936  833]
 [ 234  579]]
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      8769
           1       0.41      0.71      0.52       813

    accuracy                           0.89      9582
   macro avg       0.69      0.81      0.73      9582
weighted avg       0.92      0.89      0.90      9582



#### Oversampling

In [7]:
oversampler = RandomOverSampler(random_state=42) # Oversampling para balancear as classes
X_over, y_over = oversampler.fit_resample(X_train, y_train_bin) # aplica o oversampling que aumenta a classe minoritária
modelo_over = GradientBoostingClassifier()
avaliar_modelo(modelo_over, X_over, y_over, X_test, y_test_bin, "Modelo com Oversampling")


=== Modelo com Oversampling ===
Matriz de Confusão:
[[8233  536]
 [ 275  538]]
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.97      0.94      0.95      8769
           1       0.50      0.66      0.57       813

    accuracy                           0.92      9582
   macro avg       0.73      0.80      0.76      9582
weighted avg       0.93      0.92      0.92      9582



#### Alteração de limiar

In [8]:
modelo_limiar = GradientBoostingClassifier()
modelo_limiar.fit(X_train, y_train_bin)
probas = modelo_limiar.predict_proba(X_test)[:, 1]
y_pred_limiar = (probas >= 0.3).astype(int)  # limiar ajustado (ex: 0.3)
print("\n=== Modelo com Alteração de Limiar (0.3) ===")
print("Matriz de Confusão:")
print(confusion_matrix(y_test_bin, y_pred_limiar))
print("Relatório de Classificação:")
print(classification_report(y_test_bin, y_pred_limiar, zero_division=0))

c:\Users\joaopjpc\Desktop\Machine-Learning-\.MLvenv\lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



=== Modelo com Alteração de Limiar (0.3) ===
Matriz de Confusão:
[[8570  199]
 [ 390  423]]
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      8769
           1       0.68      0.52      0.59       813

    accuracy                           0.94      9582
   macro avg       0.82      0.75      0.78      9582
weighted avg       0.93      0.94      0.93      9582

